# 0. Imports

In [5]:
import pandas as pd
pd.options.display.float_format = '{:.2f}'.format
import numpy as np
import yfinance as yf

import warnings
warnings.filterwarnings("ignore")

import MyCustomLibrary.main as main
import MyCustomLibrary.backtester as bt
import MyCustomLibrary.backtester_tools as tools
import MyCustomLibrary.features as my_ta

# 1. System Description & Goals

Buy strength on a bull market.

**Bull Market** == price > MA

**Bear Market** == price < MA

**Entry:** Buy when price crosses over MA

**Exits:** Exit when price crosses down MA

**System Goals:**

    - CAGR    > 20%
    - MDD     < 40%
    - Calmar  > 1

# 2. Hypothesis and Research Questions
    Select trading model criteria.
        Markets Selection
        Time Frame
        Indicators


When market is trending, it will keep trending. So it is a good idea to follow teh trend.

# 3. Gather and Preprocess Data

In [6]:
# SPY – SPDR S&P 500 ETF
# VEU – Vanguard FTSE All-World ex-US ETF
# BIL – SPDR 1–3 Month Treasury Bill ETF

tickers = ['SPY', 'VEU', 'BIL']

In [7]:
quotes = yf.download(tickers, start='2005-01-01', progress=False, auto_adjust=False)
quotes = quotes.swaplevel(axis='columns').sort_index(axis='columns')
quotes.dropna(inplace=True)

quotes

Ticker           BIL                                          SPY         \
Price      Adj Close Close  High   Low  Open     Volume Adj Close  Close   
Date                                                                       
2007-05-30     71.61 91.60 91.60 91.50 91.58    1550.00    108.34 153.48   
2007-05-31     71.61 91.60 91.60 91.58 91.58   11200.00    108.22 153.32   
2007-06-01     71.62 91.62 91.62 91.62 91.62    1450.00    108.76 154.08   
2007-06-04     71.64 91.64 91.64 91.64 91.64    1050.00    108.77 154.10   
2007-06-05     71.64 91.64 91.68 91.64 91.66    3750.00    108.34 153.49   
...              ...   ...   ...   ...   ...        ...       ...    ...   
2026-02-12     91.47 91.47 91.48 91.47 91.48 6366300.00    681.27 681.27   
2026-02-13     91.51 91.51 91.51 91.50 91.50 9832500.00    681.75 681.75   
2026-02-17     91.52 91.52 91.52 91.51 91.52 6710000.00    682.85 682.85   
2026-02-18     91.53 91.53 91.53 91.52 91.52 8416600.00    686.29 686.29   
2026-02-19     91.53 91.53 91.54 91.53 91.53 2902377.00    683.01 683.01   

Ticker                                           VEU                          \
Price        High    Low   Open     Volume Adj Close Close  High   Low  Open   
Date                                                                           
2007-05-30 153.54 151.34 151.46  129013600     33.06 56.33 56.35 55.60 55.79   
2007-05-31 153.89 153.12 153.67  114866700     33.23 56.62 56.83 56.60 56.78   
2007-06-01 154.40 153.51 153.88  107771700     33.55 57.17 57.17 56.93 56.97   
2007-06-04 154.39 153.48 153.54   78008800     33.62 57.28 57.40 57.09 57.40   
2007-06-05 153.90 152.86 153.74  126917900     33.42 56.94 57.44 56.70 57.39   
...           ...    ...    ...        ...       ...   ...   ...   ...   ...   
2026-02-12 695.35 680.37 694.24  118829000     80.26 80.26 81.31 80.05 81.26   
2026-02-13 686.28 677.52 681.69   96267500     80.48 80.48 80.60 79.66 80.12   
2026-02-17 684.94 675.78 680.14   81354700     80.34 80.34 80.51 79.45 79.82   
2026-02-18 689.15 682.83 684.02   73434200     80.59 80.59 81.01 80.39 80.54   
2026-02-19 686.18 681.55 683.84   35869098     80.24 80.24 80.34 79.88 80.07   

Ticker                 
Price          Volume  
Date                   
2007-05-30  146300.00  
2007-05-31  102200.00  
2007-06-01  138100.00  
2007-06-04  115200.00  
2007-06-05  134000.00  
...               ...  
2026-02-12 4733300.00  
2026-02-13 4882400.00  
2026-02-17 4398100.00  
2026-02-18 3593600.00  
2026-02-19 1807229.00  

[4712 rows x 18 columns]

In [8]:
from MyCustomLibrary.main import resample_df_ohlc_volume


monthly_quotes = resample_df_ohlc_volume(quotes, 'BM')
monthly_quotes

BIL                                   SPY                       \
Price       Open  High   Low Close       Volume   Open   High    Low  Close   
Date                                                                          
2007-05-31 91.58 91.60 91.50 91.60     12750.00 151.46 153.89 151.34 153.32   
2007-06-29 91.62 92.00 91.62 92.00    145050.00 153.88 154.40 148.06 150.43   
2007-07-31 91.62 92.00 91.54 91.96    446600.00 150.87 155.53 145.04 145.72   
2007-08-31 91.58 92.10 91.48 91.98   1195200.00 145.18 150.59 137.00 147.59   
2007-09-28 91.62 91.94 91.54 91.80    823300.00 147.45 154.39 144.33 152.58   
...          ...   ...   ...   ...          ...    ...    ...    ...    ...   
2025-10-31 91.46 91.77 91.45 91.76 230337700.00 663.17 689.70 652.84 682.06   
2025-11-28 91.47 91.73 91.47 91.72 187758800.00 685.67 685.80 650.85 683.39   
2025-12-31 91.46 91.63 91.26 91.38 178999000.00 678.81 691.66 671.20 681.92   
2026-01-30 91.42 91.65 91.41 91.65 174239400.00 685.71 697.84 676.57 691.97   
2026-02-27 91.39 91.54 91.38 91.53 120433777.00 689.58 697.14 675.78 683.01   

                         VEU                                
Price           Volume  Open  High   Low Close      Volume  
Date                                                        
2007-05-31   243880300 55.79 56.83 55.60 56.62   248500.00  
2007-06-29  3502885400 56.97 58.44 55.02 57.00  2715400.00  
2007-07-31  3988676200 57.20 59.70 55.33 56.34  3726200.00  
2007-08-31  6195918600 55.95 64.51 50.40 56.23  4181400.00  
2007-09-28  2967009200 56.05 59.91 54.53 59.69  3751200.00  
...                ...   ...   ...   ...   ...         ...  
2025-10-31  1781815100 71.71 73.74 70.09 72.66 52387800.00  
2025-11-28  1664668300 72.88 73.99 70.28 72.97 52578700.00  
2025-12-31  1656188700 72.72 74.28 72.56 73.56 58438400.00  
2026-01-30  1600537800 74.54 79.36 74.28 77.73 82886200.00  
2026-02-27  1116313398 77.53 81.31 77.28 80.24 52870529.00  

[226 rows x 15 columns]

#### Data prep

In [70]:
lookbacks = [3, 6, 9, 12]

def df_add_strategy_factors(quotes_df: pd.DataFrame, lookbacks: list, append_df: bool=True) -> pd.DataFrame:
    ''' 
        Input: Feed a Multi Level DataFrame with several assets prices
        Output: Get the same Data Frame plus Several indicators for each symbol
    '''
    symbols = quotes_df.columns.get_level_values(0).unique()
    result_df = pd.DataFrame()

    for symbol in symbols:
        close_series = quotes_df[(symbol, 'Close')]

        new_columns = {}

        for lookback in lookbacks:
            # new_columns[f'ROC_{lookback}'] = my_ta.get_ROC(close_series, lookback)
            new_columns[f'ROC_{lookback}'] = my_ta.get_ROC(close_series, (lookback-1)).shift(1)

        symbol_df = pd.DataFrame(new_columns)
        symbol_df.columns = pd.MultiIndex.from_product([[symbol], symbol_df.columns])
        result_df = pd.concat([result_df, symbol_df], axis=1)
        
    if append_df:
        result_df = pd.concat([quotes_df, result_df], axis=1)
        
    return result_df

main.timer.start()
screened_df = df_add_strategy_factors(quotes_df=monthly_quotes, lookbacks=lookbacks, append_df=True)
main.timer.stop()

screened_df.tail(2)


[*] Total runtime: 8.30 ms


BIL                                   SPY                       \
            Open  High   Low Close       Volume   Open   High    Low  Close   
Date                                                                          
2026-01-30 91.42 91.65 91.41 91.65 174239400.00 685.71 697.84 676.57 691.97   
2026-02-27 91.39 91.54 91.38 91.53 120433777.00 689.58 697.14 675.78 683.01   

                        ...   BIL          SPY                      VEU        \
                Volume  ... ROC_9 ROC_12 ROC_3 ROC_6 ROC_9 ROC_12 ROC_3 ROC_6   
Date                    ...                                                     
2026-01-30  1600537800  ... -0.00  -0.00 -0.00  0.08  0.23   0.13  0.01  0.10   
2026-02-27  1116313398  ... -0.00  -0.00  0.01  0.07  0.17   0.16  0.07  0.12   

                         
           ROC_9 ROC_12  
Date                     
2026-01-30  0.18   0.24  
2026-02-27  0.19   0.28  

[2 rows x 27 columns]

In [71]:
screened_df.dropna(inplace=True)
screened_df.tail(15)

BIL                                   SPY                       \
            Open  High   Low Close       Volume   Open   High    Low  Close   
Date                                                                          
2024-12-31 91.47 91.67 91.29 91.43 155048200.00 602.97 609.07 580.91 586.08   
2025-01-31 91.45 91.76 91.44 91.75 157271100.00 589.39 610.78 575.35 601.82   
2025-02-28 91.45 91.73 91.44 91.72 152321000.00 592.67 613.23 582.44 594.18   
2025-03-31 91.45 91.74 91.44 91.73 278655900.00 596.18 597.34 546.87 559.39   
2025-04-30 91.43 91.73 91.42 91.72 381763600.00 557.45 567.42 481.80 554.54   
2025-05-30 91.43 91.75 91.42 91.74 247955000.00 560.37 595.54 556.04 589.39   
2025-06-30 91.45 91.74 91.44 91.73 179353200.00 587.76 619.22 585.06 617.85   
2025-07-31 91.44 91.76 91.43 91.75 170675800.00 616.36 639.85 615.52 632.08   
2025-08-29 91.47 91.78 91.46 91.77 187741200.00 626.30 649.48 619.29 645.05   
2025-09-30 91.47 91.76 91.46 91.75 179049900.00 637.50 667.34 634.92 666.18   
2025-10-31 91.46 91.77 91.45 91.76 230337700.00 663.17 689.70 652.84 682.06   
2025-11-28 91.47 91.73 91.47 91.72 187758800.00 685.67 685.80 650.85 683.39   
2025-12-31 91.46 91.63 91.26 91.38 178999000.00 678.81 691.66 671.20 681.92   
2026-01-30 91.42 91.65 91.41 91.65 174239400.00 685.71 697.84 676.57 691.97   
2026-02-27 91.39 91.54 91.38 91.53 120433777.00 689.58 697.14 675.78 683.01   

                        ...   BIL          SPY                      VEU        \
                Volume  ... ROC_9 ROC_12 ROC_3 ROC_6 ROC_9 ROC_12 ROC_3 ROC_6   
Date                    ...                                                     
2024-12-31  1059516700  ... -0.00   0.00  0.05  0.11  0.15   0.27 -0.05  0.02   
2025-01-31   995501600  ... -0.00  -0.00  0.03  0.06  0.17   0.21 -0.05 -0.05   
2025-02-28   871641300  ... -0.00  -0.00 -0.00  0.07  0.14   0.18 -0.01 -0.04   
2025-03-31  1496591400  ... -0.00  -0.00  0.01  0.04  0.09   0.14  0.06 -0.04   
2025-04-30  2237015100  ... -0.00  -0.00 -0.07 -0.02  0.02   0.11  0.02  0.01   
2025-05-30  1402172600  ... -0.00  -0.00 -0.07 -0.08 -0.02   0.05  0.03  0.04   
2025-06-30  1495410200  ... -0.00  -0.00  0.05  0.01  0.03   0.08  0.08  0.14   
2025-07-31  1479850800  ... -0.00  -0.00  0.11  0.03  0.09   0.12  0.08  0.13   
2025-08-29  1424047400  ... -0.00  -0.00  0.07  0.06  0.05   0.12  0.02  0.10   
2025-09-30  1606348500  ...  0.00  -0.00  0.04  0.15  0.10   0.12  0.03  0.14   
2025-10-31  1781815100  ...  0.00  -0.00  0.05  0.20  0.11   0.17  0.07  0.14   
2025-11-28  1664668300  ...  0.00  -0.00  0.06  0.16  0.15   0.13  0.05  0.11   
2025-12-31  1656188700  ... -0.00   0.00  0.03  0.11  0.22   0.17  0.02  0.09   
2026-01-30  1600537800  ... -0.00  -0.00 -0.00  0.08  0.23   0.13  0.01  0.10   
2026-02-27  1116313398  ... -0.00  -0.00  0.01  0.07  0.17   0.16  0.07  0.12   

                         
           ROC_9 ROC_12  
Date                     
2024-12-31  0.02   0.07  
2025-01-31  0.00   0.04  
2025-02-28 -0.00   0.04  
2025-03-31  0.03   0.03  
2025-04-30  0.01   0.06  
2025-05-30  0.01   0.05  
2025-06-30  0.04   0.11  
2025-07-31  0.12   0.12  
2025-08-29  0.11   0.08  
2025-09-30  0.21   0.10  
2025-10-31  0.20   0.19  
2025-11-28  0.20   0.21  
2025-12-31  0.20   0.27  
2026-01-30  0.18   0.24  
2026-02-27  0.19   0.28  

[15 rows x 27 columns]

# 4. Backtesting

In [72]:
# Signals generation

def buy_ticker(screened_df: pd.DataFrame, tickers: list, lookback: int, row: int):
    
    # roc1 and roc2 are the equities modules
    roc1 = screened_df[(tickers[0], f'ROC_{lookback}')].iloc[row]
    roc2 = screened_df[(tickers[1], f'ROC_{lookback}')].iloc[row]
    
    # treasuries roc is the risk free module
    #treasuries_roc = screened_df[(tickers[2], f'ROC_{lookback}')].iloc[row]
    treasuries_roc = 0.02

    if roc1 > roc2 and roc1 > treasuries_roc:
        return tickers[0]
    
    elif roc2 > roc1 and roc2 > treasuries_roc:
        return tickers[1]
    
    else:
        return tickers[2]

In [73]:
# Strategy Implementation

class DualMomentumStrategy(bt.Strategy):
    
    def init(self,
             tickers_list:list,
             lookback:int,
             **kwargs
             ):
        
        self.symbol = 'BTCUSDT'
        self.lookback = lookback
        self.tickers_list = tickers_list

    def next(self, i, record):
        
        try:
            old_ticker = buy_ticker(self.data, self.tickers_list, self.lookback, row=i-2)
            new_ticker = buy_ticker(self.data, self.tickers_list, self.lookback, row=i-1)
            
            if old_ticker == new_ticker:
                pass

            else:
                ###########################################
                ###          Sell conditions            ###
                ###########################################
                #if self.has_position(old_ticker):
                self.close(symbol=old_ticker, price=record[(old_ticker,'Open')])
                
                # else:
                #     raise Exception(f"Trying to close a position that doesn't exist: {old_ticker}")


                ###########################################
                ###          Buy conditions             ###
                ###########################################
        
                self.open_long(symbol = new_ticker, price = record[(new_ticker,'Open')])
                        

        except Exception as e:
            import traceback
            print(traceback.format_exc())

## Limited Testing

Just test if everything is working as planned.

In [ ]:
import importlib

importlib.reload(bt)
importlib.reload(tools)
importlib.reload(main)

In [74]:
df_train, df_test = main.train_test_split(screened_df, test_size=0)

main.timer.start()
backtest = bt.Backtest(DualMomentumStrategy, df_train, df_train[('SPY', 'Close')], cash=100_000, commission=0.00)
result = backtest.run(tickers_list=tickers, lookback = 12)
main.timer.stop()


[*] Total runtime: 60.95 ms


In [75]:
result.stats

,Backtest
Start Period,2008-05-30
End Period,2026-02-27
Duration (Months),213
,
CAGR %,7.52
Ann. Vola %,69.92
Max Drawdown %,-19.92
VaR 5%,-5.83
Time in Market %,98.12
Sharpe,0.11


In [76]:
result.trades

,symbol,open_date,close_date,bars_held,open_price,close_price,position_size,profit_loss,change_pct,trade_commission,cumulative_return,current_open_positions,why_exit
0,BIL,2008-06-30,2009-11-30,17,91.62,91.76,1091.46,152.80,0.15,0.00,100152.80,1,None
1,VEU,2009-11-30,2010-06-30,7,42.22,38.46,2372.16,-8919.35,-8.91,0.00,91233.46,1,None
2,SPY,2010-06-30,2010-10-29,4,108.35,114.99,842.03,5591.05,6.13,0.00,96824.51,1,None
3,BIL,2010-10-29,2010-11-30,1,91.70,91.72,1055.88,21.12,0.02,0.00,96845.63,1,None
4,SPY,2010-11-30,2011-05-31,6,119.07,137.07,813.35,14640.31,15.12,0.00,111485.94,1,None
5,VEU,2011-05-31,2011-08-31,3,52.05,49.17,2141.90,-6168.68,-5.53,0.00,105317.27,1,None
6,SPY,2011-08-31,2011-11-30,3,130.84,122.03,804.93,-7091.45,-6.73,0.00,98225.82,1,None
7,BIL,2011-11-30,2011-12-30,1,91.70,91.66,1071.16,-42.84,-0.04,0.00,98182.98,1,None
8,SPY,2011-12-30,2012-01-31,1,124.85,127.76,786.41,2288.45,2.33,0.00,100471.43,1,None
9,BIL,2012-01-31,2012-04-30,3,91.66,91.64,1096.13,-21.93,-0.02,0.00,100449.50,1,None


In [77]:
result.open_positions

,symbol,open_date,last_date,bars_holding,open_price,last_price,position_size,profit_loss,change_pct,current_value
0,VEU,2025-11-28,2026-02-27,5,72.88,80.24,4516.02,33244.26,10.10,362371.86


In [78]:
main.plot_strategy_performance(result.returns, result.benchmark, y_log_scale=False)

In [79]:
tools.strategy_performance_metrics(pd.concat([result.returns, result.benchmark], axis=1))

,Strategy Backtest,Benchmark
CAGR %,7.52,9.32
Ann. Vola %,69.92,86.71
Max Drawdown %,-19.92,-47.32
VaR 5%,-5.83,-7.93
Time in Market %,98.12,99.53
Sharpe,0.11,0.11
Sortino,0.14,0.14
Calmar,0.38,0.20
Efficiency,0.08,0.09


In [30]:
# tools.plot_trades_signals(df_train['SPY'][['Close']], result.trades)

## Optimization Training Data

    It is important that the basic buy-and-sell rules are successful across a range of calculation periods and similar markets. If this fails, then you need to rethink your idea. I don’t believe in adding rules to make the strategy more complex in the hope that one rule will turn a losing idea into a profitable one. It is necessary that the basic concept is robust before moving forward. Then we can try to make it better. Kaufman

In [ ]:
import importlib

importlib.reload(bt)
importlib.reload(tools)
importlib.reload(main)

In [ ]:
lookbacks

In [ ]:
nr_confirmations = np.arange(2,4)
nr_confirmations

In [ ]:
df_train, df_test = main.train_test_split(screened_df, test_size=0.20)
optimize = bt.Optimization(TrendStrategy, df_train, df_train[('BTCUSDT', 'Close')], cash=100_000, commission=0.001)

values = [
    lookbacks,
    nr_confirmations
    ]

main.timer.start()
optimize.run(values)
main.timer.stop()

In [ ]:
backtest_df = optimize.equity_curves()
backtest_df.tail(2)

In [ ]:
strats_performance = tools.strategy_performance_metrics(backtest_df, steps_per_year=(24*365))
strats_performance.sort_values(by='CAGR %', ascending=False, axis=1)

In [ ]:
# Parameter Sensitivity
tools.parameter_sensitivity_bar_plot(strats_performance, 'CAGR %')

In [ ]:
# % profitable Backtests
# -1 to remove Benchmark from equation

(len(strats_performance.loc['CAGR %'][strats_performance.loc['CAGR %']>0])-1) / (len(strats_performance.loc['CAGR %'])-1)

#### It is not a robust Strategy. Avoid!

In [ ]:
tools.parameter_sensitivity_bar_plot(strats_performance, 'Calmar')

In [ ]:
tools.metrics_time_window_analysis(backtest_df)

In [ ]:
tools.plot_matrix(strats_performance)

In [ ]:
# main.plot_performance_drawdown(backtest_df['lookback_30'])

In [ ]:
main.plot_comparative_graph(backtest_df)

In [ ]:
main.correlation_map(backtest_df)

### Agile Phases Analysis

Agile System Construction

    0. add entries & exits
    1. add filters
    2. add Market Selection
    2. add stop loss
    3. position sizing

    In each iteration compare results with previous iterations (Is there an improvement or not?)

Agile Phases Description

 - ph1: Entry > 50; Exit < 50; Lookback 50
 - ph2: Entry > 50 2 days; Exit < 50 2 days; Lookback 50
 - ph3: Entry fgiMA20 > fgiMA50; Exit fgiMA20 < fgiMA50
 - ph4: Entry fgiMA20 > fgiMA50 2 days; Exit fgiMA20 < fgiMA50 2 days

#### Agile Phases Save

In [ ]:
import os

_strat_number = "049"
strategy_returns = result.returns
_file_name = strategy_returns.name = "BTC_H_BBands_ph0"
_path = f"agile_phases_dump/{_strat_number}"

os.makedirs(f"{_path}", exist_ok=True)
strategy_returns.to_csv(f"{_path}/{_file_name}.csv")
print("Done!")

#### Read all Equity Curves Phases

In [ ]:
import glob
from pathlib import Path

files = Path(_path).glob('*.csv')  # .rglob to get subdirectories

all_df = []
for file in files:
    # Leitura do CSV
    data = pd.read_csv(file, index_col=[0], parse_dates=True)
    
    # Adicionar o DataFrame à lista
    all_df.append(data)


all_agile_phases_df = pd.concat(all_df, axis=1, join='outer')
all_agile_phases_df['Benchmark'] = result.benchmark
all_agile_phases_df.head(3)

In [ ]:
main.plot_comparative_graph(all_agile_phases_df)

In [ ]:
tools.strategy_performance_metrics(all_agile_phases_df)

In [ ]:
tools.metrics_time_window_analysis(all_agile_phases_df)

In [ ]:
main.correlation_map(all_agile_phases_df)

## Out of Sample Validation

In [ ]:
lookbacks

In [ ]:
stds

In [ ]:
main.timer.start()
backtest = bt.Backtest(TrendStrategy, df_test, df_test[('BTCUSDT', 'Close')], cash=100_000, commission=0.00)
result = backtest.run(320, 1.5)
main.timer.stop()

In [ ]:
result.stats

In [ ]:
main.plot_performance_drawdown(result.returns, result.benchmark)

##### Is it significantly different from 0?

In [ ]:
main.compute_t_test(result.returns)

##### Does it pass Monte Carlo Permutation?

In [ ]:
mcpyt = main.monte_carlo_permutation_yt(result.returns, nr_simulations=5000)

In [ ]:
main.mcpyt_analysis(mcpyt, result.returns)

Statistically speaking this is a sound strategy! It has proven itself against 0 returns, and has passed Monte Carlo Permutation test, proving that it has an edge, since it's Max DD was lower that 95% CI.

##### Returns Analysis

In [ ]:
tools.strategy_performance_metrics(pd.concat([result.returns, result.benchmark], axis=1), steps_per_year=(365*24))

In [ ]:
main.monthly_pos_neg_returns(result.returns)

##### Trades Analysis

In [ ]:
tools.metrics_trades(result.trades)

In [ ]:
tools.plot_trades_signals(df_train['BTCUSDT'][['Close', 'BOLL_UP_40_2.5']], result.trades)

In [ ]:
clipped_returns = main.cap_series(result.trades['change_pct'], percentile=0.005)
ax2 = clipped_returns.plot.hist(bins=50, title="Trades Return Distribution", figsize=(10, 5))

In [ ]:
exit_types_df = ((result.trades.groupby(['why_exit']).size()/len(result.trades))*100).sort_values()

exit_types_df.plot.barh(title='Exits Types (%)', figsize=(10,5));

In [ ]:
(result.trades.groupby(['symbol']).size()/len(result.trades)).plot.barh(title='Symbols traded (%)', figsize=(10,5));

In [ ]:
x4 = result.trades['current_open_positions'].plot(title='Open Positions', figsize=(10, 5))

In [ ]:
ax1 = result.trades['bars_held'].plot.hist(bins=50, title="Bars Held Distribution", figsize=(10, 5))
ax1.axvline(result.trades['bars_held'].mean(), color='red', linestyle='dashed', linewidth=2)
print("Avg. Bars Held:", int(result.trades['bars_held'].mean()))

In [ ]:
# Bar Duration VS Percent Profit
import plotly.express as px

# Fazer scatter plot
fig = px.scatter(result.trades, 
                 x='bars_held', 
                 y='change_pct', 
                 width=900, 
                 height=500, 
                 trendline="ols", 
                 trendline_color_override="black", 
                 title='Bar Duration VS Percent Profit',
                 template='seaborn'
                 )

fig.show()